# CodeT5+ for Code Classification
## Efficient Encoder-Only Approach

In [2]:
import os
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer, 
    T5EncoderModel,
    Trainer, 
    TrainingArguments,
    EvalPrediction
)
from tqdm.auto import tqdm
import numpy as np

/home/info-sec-lab/BTP/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Configuration
MODEL_NAME = "Salesforce/codet5p-220m"
NUM_LABELS = 2
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
EPOCHS = 3
MAX_LENGTH = 512

base_text_path = "../Text_Files/"

In [4]:
class CodeT5ForSequenceClassification(nn.Module):
    """
    Custom CodeT5+ model for sequence classification using only the encoder
    """
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.encoder.config.d_model, num_labels)
        self.num_labels = num_labels
        
    def forward(self, input_ids, attention_mask=None, labels=None):
        # Get encoder outputs
        outputs = self.encoder(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Use mean pooling over sequence dimension
        sequence_output = outputs.last_hidden_state
        
        # Apply attention mask for mean pooling
        if attention_mask is not None:
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(sequence_output.size()).float()
            sum_embeddings = torch.sum(sequence_output * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            pooled_output = sum_embeddings / sum_mask
        else:
            pooled_output = sequence_output.mean(dim=1)
        
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
        
        return {'loss': loss, 'logits': logits}

In [5]:
class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [6]:
def load_and_preprocess_data(base_path, tokenizer, folder_name):
    """
    Load and preprocess code data from text files
    """
    codes = []
    labels = []
    
    for label_dir in ["Label_0", "Label_1"]:
        current_path = os.path.join(base_path, folder_name, label_dir)
        if not os.path.exists(current_path):
            print(f"Warning: Directory {current_path} not found. Skipping.")
            continue
            
        for filename in os.listdir(current_path):
            if filename.endswith(".txt"):
                filepath = os.path.join(current_path, filename)
                try:
                    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                        code_content = f.read().strip()
                    if code_content:  # Only add non-empty files
                        codes.append(code_content)
                        labels.append(0 if label_dir == "Label_0" else 1)
                except Exception as e:
                    print(f"Error reading {filepath}: {e}")
                    continue

    print(f"Loaded {len(codes)} samples from {folder_name}")
    
    if len(codes) == 0:
        return CodeDataset({'input_ids': torch.tensor([]), 'attention_mask': torch.tensor([])}, [])

    # Tokenize with proper T5 formatting
    encodings = tokenizer(
        codes,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        add_special_tokens=True,
        return_tensors="pt"
    )

    return CodeDataset(encodings, labels)

In [7]:
# Initialize tokenizer and model
print("Initializing tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Add padding token if it doesn't exist
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = CodeT5ForSequenceClassification(MODEL_NAME, NUM_LABELS)

# Print model architecture
print("Model architecture:")
print(f"Encoder parameters: {sum(p.numel() for p in model.encoder.parameters()):,}")
print(f"Classifier parameters: {sum(p.numel() for p in model.classifier.parameters()):,}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Initializing tokenizer and model...
Model architecture:
Encoder parameters: 109,607,040
Classifier parameters: 1,538
Total parameters: 109,608,578


In [11]:
# Load training data
print("\nLoading training data...")
train_dataset = load_and_preprocess_data(base_text_path, tokenizer, "Train")

if len(train_dataset) == 0:
    raise ValueError("No training data found! Please check your data paths.")

print(f"Training dataset size: {len(train_dataset)}")
print(f"Label distribution: {np.bincount(train_dataset.labels)}")


Loading training data...
Loaded 6190 samples from Train
Training dataset size: 6190
Label distribution: [3007 3183]


In [8]:
def compute_metrics(p: EvalPrediction):
    """
    Compute metrics for evaluation
    """
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    preds = np.argmax(preds, axis=1)
    
    accuracy = accuracy_score(p.label_ids, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        p.label_ids, preds, average='binary', zero_division=0
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [9]:
# Training arguments
training_args = TrainingArguments(
    output_dir="../checkpoints/ct5p_only",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="no",  # Change to "steps" or "epoch" if you have validation data
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to=None,  # Disable wandb/tensorboard if not needed
    dataloader_pin_memory=False,
    save_safetensors=False,
)

In [12]:
# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics,
)

In [13]:
# Start training
print("Starting training...")
trainer.train()

Starting training...


Step,Training Loss
50,0.683800
100,0.667500
150,0.651100
200,0.602400
250,0.489500
300,0.414300
350,0.382100
400,0.386400
450,0.322200
500,0.338200


TrainOutput(global_step=2322, training_loss=0.2527507224748301, metrics={'train_runtime': 831.4601, 'train_samples_per_second': 22.334, 'train_steps_per_second': 2.793, 'total_flos': 0.0, 'train_loss': 0.2527507224748301, 'epoch': 3.0})

In [ ]:
# Save the final model
trainer.save_model("../checkpoints/ct5p_only")
tokenizer.save_pretrained("../checkpoints/ct5p_only")
print("Training completed and model saved!")

Training completed and model saved!


# Model Evaluation

In [14]:
# Load the trained model for evaluation
def load_trained_model(model_path):
    """Load trained model and tokenizer"""
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # Initialize model architecture
    model = CodeT5ForSequenceClassification(MODEL_NAME, NUM_LABELS)
    
    # Load trained weights
    model.load_state_dict(torch.load(os.path.join(model_path, "pytorch_model.bin")))
    
    return model, tokenizer

# For simplicity, we'll use the trainer's model directly
print("Model ready for evaluation!")

Model ready for evaluation!


In [15]:
# Evaluate on test datasets
print("\nEvaluating on test datasets...")

results = {}

for i in range(10):
    test_folder = f"Test_{i}"
    print(f"\nLoading test data for {test_folder}...")
    
    test_dataset = load_and_preprocess_data(base_text_path, tokenizer, test_folder)
    
    if len(test_dataset) > 0:
        # Get predictions
        predictions = trainer.predict(test_dataset)
        preds = np.argmax(predictions.predictions, axis=1)
        true_labels = test_dataset.labels

        # Calculate metrics
        accuracy = accuracy_score(true_labels, preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            true_labels, preds, average='binary', zero_division=0
        )

        results[test_folder] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'samples': len(test_dataset)
        }

        print(f"Results for {test_folder}:")
        print(f"  Samples: {len(test_dataset)}")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-Score: {f1:.4f}")
    else:
        print(f"No data found for {test_folder}. Skipping evaluation.")
        results[test_folder] = None


Evaluating on test datasets...

Loading test data for Test_0...
Loaded 1002 samples from Test_0


Results for Test_0:
  Samples: 1002
  Accuracy: 0.8673
  Precision: 0.8710
  Recall: 0.8623
  F1-Score: 0.8666

Loading test data for Test_1...
Loaded 1002 samples from Test_1


Results for Test_1:
  Samples: 1002
  Accuracy: 0.8263
  Precision: 0.8045
  Recall: 0.8623
  F1-Score: 0.8324

Loading test data for Test_2...
Loaded 1015 samples from Test_2


Results for Test_2:
  Samples: 1015
  Accuracy: 0.8236
  Precision: 0.7970
  Recall: 0.8623
  F1-Score: 0.8284

Loading test data for Test_3...
Loaded 1002 samples from Test_3


Results for Test_3:
  Samples: 1002
  Accuracy: 0.8114
  Precision: 0.7826
  Recall: 0.8623
  F1-Score: 0.8205

Loading test data for Test_4...
Loaded 1002 samples from Test_4


...

Results for Test_5:
  Samples: 1002
  Accuracy: 0.9002
  Precision: 0.9330
  Recall: 0.8623
  F1-Score: 0.8963

Loading test data for Test_6...
Loaded 1002 samples from Test_6


Results for Test_6:
  Samples: 1002
  Accuracy: 0.7665
  Precision: 0.7236
  Recall: 0.8623
  F1-Score: 0.7869

Loading test data for Test_7...
Loaded 1002 samples from Test_7


Results for Test_7:
  Samples: 1002
  Accuracy: 0.7136
  Precision: 0.6646
  Recall: 0.8623
  F1-Score: 0.7507

Loading test data for Test_8...
Loaded 1002 samples from Test_8


Results for Test_8:
  Samples: 1002
  Accuracy: 0.7665
  Precision: 0.7236
  Recall: 0.8623
  F1-Score: 0.7869

Loading test data for Test_9...
Loaded 1002 samples from Test_9


Results for Test_9:
  Samples: 1002
  Accuracy: 0.7136
  Precision: 0.6646
  Recall: 0.8623
  F1-Score: 0.7507


In [16]:
# Print summary of results
print("\n" + "="*50)
print("SUMMARY OF RESULTS")
print("="*50)

valid_results = {k: v for k, v in results.items() if v is not None}

if valid_results:
    avg_accuracy = np.mean([r['accuracy'] for r in valid_results.values()])
    avg_precision = np.mean([r['precision'] for r in valid_results.values()])
    avg_recall = np.mean([r['recall'] for r in valid_results.values()])
    avg_f1 = np.mean([r['f1'] for r in valid_results.values()])
    
    print(f"\nAverage across all test sets:")
    print(f"Accuracy:  {avg_accuracy:.4f}")
    print(f"Precision: {avg_precision:.4f}")
    print(f"Recall:    {avg_recall:.4f}")
    print(f"F1-Score:  {avg_f1:.4f}")
    
    # Save results to CSV
    results_df = pd.DataFrame.from_dict(valid_results, orient='index')
    results_df.to_csv("codet5_classification_results.csv")
    print("\nDetailed results saved to 'codet5_classification_results.csv'")
else:
    print("No valid results to display.")


SUMMARY OF RESULTS

Average across all test sets:
Accuracy:  0.7962
Precision: 0.7697
Recall:    0.8623
F1-Score:  0.8111

Detailed results saved to 'codet5_classification_results.csv'


# Inference Example

In [ ]:
def predict_single_code_snippet(code_snippet, model, tokenizer):
    """
    Predict label for a single code snippet
    """
    model.eval()
    
    # Tokenize
    inputs = tokenizer(
        code_snippet,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )
    
    # Move to device
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs['logits']
        pred = torch.argmax(logits, dim=1).cpu().item()
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]
    
    return pred, probabilities

# Example usage
test_code = """
def example_function(x):
    return x + 1
"""

prediction, probs = predict_single_code_snippet(test_code, model, tokenizer)
print(f"Prediction: Label {prediction}")
print(f"Probabilities: [Label 0: {probs[0]:.4f}, Label 1: {probs[1]:.4f}]")

In [ ]:
# Clean up
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("Execution completed!")